In [101]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8")


In [102]:
market_options = {
    "NSE (India)": "^NSEI",
    "BSE (India)": "^BSESN",
    "S&P 500 (US)": "^GSPC",
    "NASDAQ (US)": "^IXIC"
}

selected_market = market_options[user_choice]

df = yf.download(selected_market, start="1990-01-01")
df = df[['Close']]
df.dropna(inplace=True)


NameError: name 'user_choice' is not defined

In [ ]:
# Log Returns
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))

# Rolling Volatility (20-day)
df['Rolling_Vol'] = df['Log_Return'].rolling(20).std()

df.dropna(inplace=True)


In [ ]:
scaler = StandardScaler()

train = df[:'2018-12-31'].copy()
test = df['2019-01-01':].copy()

scaler.fit(train[['Log_Return','Rolling_Vol']])

train_scaled = scaler.transform(train[['Log_Return','Rolling_Vol']])
test_scaled = scaler.transform(test[['Log_Return','Rolling_Vol']])

train['Scaled_Log_Return'] = train_scaled[:,0]
train['Scaled_Rolling_Vol'] = train_scaled[:,1]

test['Scaled_Log_Return'] = test_scaled[:,0]
test['Scaled_Rolling_Vol'] = test_scaled[:,1]


In [ ]:

X_train = train[['Scaled_Log_Return','Scaled_Rolling_Vol']].values
X_test  = test[['Scaled_Log_Return','Scaled_Rolling_Vol']].values


In [ ]:

model = GaussianHMM(
    n_components=3,
    covariance_type="diag",
    n_iter=1000,
    random_state=42
)

model.fit(X_train)

In [ ]:

train.loc[:, 'Regime'] = model.predict(X_train)
test.loc[:, 'Regime'] = model.predict(X_test)

df = pd.concat([train, test])

In [ ]:
variances = model.covars_[:, 0, 0]  # Variance of Log_Return
crisis_regime = np.argmax(variances)


In [ ]:

df['Regime_Shifted'] = df['Regime'].shift(1)

df['Strategy_Return'] = np.where(
    df['Regime_Shifted'] == crisis_regime,
    0,
    df['Log_Return']
)

df.dropna(inplace=True)

In [ ]:

df['Market_Cum_Return'] = np.exp(df['Log_Return'].cumsum())
df['Strategy_Cum_Return'] = np.exp(df['Strategy_Return'].cumsum())

In [ ]:

def sharpe_ratio(returns):
    return np.sqrt(252) * returns.mean() / returns.std()

In [ ]:
def max_drawdown(cum_returns):
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    return drawdown.min()

In [ ]:


market_sharpe = sharpe_ratio(df['Log_Return'])
strategy_sharpe = sharpe_ratio(df['Strategy_Return'])

market_dd = max_drawdown(df['Market_Cum_Return'])
strategy_dd = max_drawdown(df['Strategy_Cum_Return'])

In [ ]:
print("PERFORMANCE METRICS")

print("Market Sharpe:", market_sharpe)
print("Strategy Sharpe:", strategy_sharpe)
print("Market Max Drawdown:", market_dd)
print("Strategy Max Drawdown:", strategy_dd)



In [ ]:
market_return = df['Market_Cum_Return'].iloc[-1]
strategy_return = df['Strategy_Cum_Return'].iloc[-1]

print("Final Market Return:", market_return)
print("Final Strategy Return:", strategy_return)

In [ ]:

plt.figure(figsize=(14,6))
plt.plot(df['Close'], color='black', linewidth=1)

for regime in df['Regime'].unique():
    mask = df['Regime'] == regime
    plt.scatter(df.index[mask], df['Close'][mask], s=5, label=f'Regime {regime}')

plt.title("Financial Market Regimes Detected by HMM")
plt.legend()
plt.show()

In [ ]:

plt.figure(figsize=(14,6))
plt.plot(df['Market_Cum_Return'], label="Market")
plt.plot(df['Strategy_Cum_Return'], label="Regime Strategy")
plt.title("Market vs Regime-Based Strategy")
plt.legend()
plt.show()

In [ ]:
# ==========================================
# WALK FORWARD VALIDATION (ROBUSTNESS TEST)
# ==========================================

from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Performance Functions
def sharpe_ratio(returns):
    if returns.std() == 0:
        return 0
    return np.sqrt(252) * returns.mean() / returns.std()

def max_drawdown(cum_returns):
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    return drawdown.min()

# Walk-forward splits (based on your data starting ~2008)
splits = [
    ('2012-12-31', '2016-12-31'),
    ('2016-12-31', '2019-12-31'),
    ('2019-12-31', '2022-12-31'),
    ('2022-12-31', '2026-01-01')
]

results = []

for train_end, test_end in splits:

    print(f"\nTrain End: {train_end} | Test End: {test_end}")

    train = df[:train_end].copy()
    test = df[train_end:test_end].copy()

    # Safety check
    if len(train) < 100 or len(test) < 50:
        print("Not enough data. Skipping this split.")
        continue

    # Scaling (fit only on train)
    scaler = StandardScaler()
    scaler.fit(train[['Log_Return', 'Rolling_Vol']])

    train_scaled = scaler.transform(train[['Log_Return', 'Rolling_Vol']])
    test_scaled = scaler.transform(test[['Log_Return', 'Rolling_Vol']])

    X_train = train_scaled
    X_test = test_scaled

    # HMM Model
    model = GaussianHMM(
        n_components=3,
        covariance_type="diag",
        n_iter=1000,
        random_state=42
    )

    model.fit(X_train)

    train['Regime'] = model.predict(X_train)
    test['Regime'] = model.predict(X_test)

    combined = pd.concat([train, test])

    # Identify crisis regime (highest variance in returns)
    variances = model.covars_[:, 0, 0]
    crisis_regime = np.argmax(variances)

    combined['Regime_Shifted'] = combined['Regime'].shift(1)

    combined['Strategy_Return'] = np.where(
        combined['Regime_Shifted'] == crisis_regime,
        0,
        combined['Log_Return']
    )

    combined.dropna(inplace=True)

    combined['Market_Cum_Return'] = np.exp(combined['Log_Return'].cumsum())
    combined['Strategy_Cum_Return'] = np.exp(combined['Strategy_Return'].cumsum())

    # Performance Metrics
    market_sharpe = sharpe_ratio(combined['Log_Return'])
    strategy_sharpe = sharpe_ratio(combined['Strategy_Return'])

    market_dd = max_drawdown(combined['Market_Cum_Return'])
    strategy_dd = max_drawdown(combined['Strategy_Cum_Return'])

    results.append([
        train_end,
        test_end,
        market_sharpe,
        strategy_sharpe,
        market_dd,
        strategy_dd
    ])

# Results DataFrame
results_df = pd.DataFrame(results, columns=[
    "Train_End",
    "Test_End",
    "Market_Sharpe",
    "Strategy_Sharpe",
    "Market_Drawdown",
    "Strategy_Drawdown"
])

print("\n=================================")
print("WALK FORWARD RESULTS")
print("=================================")
print(results_df)